# Phase 4: Finance Domain Fine-Tuning (QLoRA)

**Dataset:** `gbharti/finance-alpaca` — 68K finance Q&A pairs (shuffled 10K subset)  
**Base model:** `mistralai/Mistral-7B-Instruct-v0.2`  
**Method:** QLoRA (4-bit quantization + LoRA adapters, r=16)  
**GPU required:** T4 (16GB) — available free on Kaggle  
**Estimated runtime:** ~50 minutes (3 epochs, 9K training samples, seq_len=512)

## What this notebook does
1. Loads and explores the finance-alpaca dataset
2. Trains a QLoRA adapter with a finance system prompt + disclaimer
3. Evaluates quality with LLM-as-judge via Groq (finance rubric)
4. Pushes the adapter to HuggingFace Hub

## Why finance fine-tuning?
Base Mistral-7B knows financial concepts but:
- Doesn't reliably apply correct formulas (P/E, EBITDA, DCF)
- Misses domain conventions (e.g. gross margin vs net margin semantics)
- Doesn't consistently flag date-sensitive data (stock prices, interest rates change daily)

Fine-tuning on finance-alpaca teaches the model to reason in financial analyst style:
define the concept → show the formula → apply it → interpret the result → note caveats.

## 0. Kaggle Setup

Before running:
1. Enable GPU: **Settings → Accelerator → GPU T4 x1** (single GPU)
2. Enable Internet: **Settings → Internet → On**
3. Add secrets: **Add-ons → Secrets** → `HF_TOKEN`, `GROQ_API_KEY`

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

## 1. Install Dependencies

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "bitsandbytes==0.45.3",
     "transformers==4.46.3",
     "peft==0.13.2",
     "trl==0.12.2",
     "accelerate==1.1.1",
     "datasets==3.0.0",
     "huggingface_hub",
     "groq",
    ],
    capture_output=True, text=True
)
print(result.stdout[-2000:] or "")
print(result.stderr[-500:] or "")
print("\nDone. Restart kernel if needed.")

In [ ]:
import transformers, peft, trl, bitsandbytes, accelerate, datasets
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")

## 2. Hyperparameters

In [ ]:
import torch

BASE_MODEL   = "mistralai/Mistral-7B-Instruct-v0.2"
DATASET_NAME = "gbharti/finance-alpaca"

# ── Dataset ─────────────────────────────────────────────────────────────────
MAX_SAMPLES     = 10_000   # shuffled subset of 68K — keeps runtime on T4 manageable
SHUFFLE_SEED    = 42
TEST_SIZE       = 0.10     # ~1K held-out for eval

# ── LoRA ────────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# ── Training ────────────────────────────────────────────────────────────────
OUTPUT_DIR          = "/kaggle/working/phase4-finance"
NUM_EPOCHS          = 3
BATCH_SIZE          = 4
GRAD_ACCUM_STEPS    = 4    # effective batch = 16
LEARNING_RATE       = 2.0e-4
MAX_SEQ_LENGTH      = 512  # finance Q&A is short — saves VRAM vs legal phases
LOGGING_STEPS       = 10
SAVE_STEPS          = 200

# ── Eval ────────────────────────────────────────────────────────────────────
MAX_EVAL_SAMPLES    = 100
DOMAIN_BASELINE     = "~3.0/5.0"
DOMAIN_TARGET       = "≥3.5/5.0"

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" \
      if torch.cuda.is_available() else "")

## 3. Load and Explore the Dataset

`gbharti/finance-alpaca` uses `instruction` / `input` / `output` — same schema as Dolly.  
The `input` field sometimes contains additional context (e.g. a balance sheet snippet).  
Many samples have empty `input`, so we handle both cases.

In [ ]:
from datasets import load_dataset

print(f"Loading {DATASET_NAME}...")
raw = load_dataset(DATASET_NAME, split="train")
print(f"Full dataset: {len(raw)} samples | Columns: {raw.column_names}")

# Shuffle and take MAX_SAMPLES subset
raw = raw.shuffle(seed=SHUFFLE_SEED).select(range(min(MAX_SAMPLES, len(raw))))
print(f"Using: {len(raw)} samples (shuffled subset)")
print()

for i in range(3):
    s = raw[i]
    print(f"─── Sample {i} ───")
    print(f"INSTRUCTION: {s['instruction'][:200]}")
    print(f"INPUT:       {s.get('input', '')[:100] or '(empty)'}")
    print(f"OUTPUT:      {s['output'][:200]}")
    print()

In [ ]:
# Dataset statistics
import numpy as np

with_context = sum(1 for s in raw if s.get('input', '').strip())
print(f"Samples with non-empty 'input' (context): {with_context} / {len(raw)} ({with_context/len(raw):.1%})")

lengths = [len(s['instruction']) + len(s.get('input', '')) + len(s['output']) for s in raw]
print(f"\nChar length stats:")
print(f"  median: {int(np.median(lengths))}")
print(f"  p90:    {int(np.percentile(lengths, 90))}")
print(f"  p99:    {int(np.percentile(lengths, 99))}")
print(f"  max_seq_length={MAX_SEQ_LENGTH} (512) fits p90: {'yes' if int(np.percentile(lengths, 90)) // 4 < MAX_SEQ_LENGTH else 'no'}")

## 4. Load Tokenizer and Format Dataset

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"
print(f"Tokenizer loaded. Vocab: {tokenizer.vocab_size}")

In [ ]:
FINANCE_SYSTEM = (
    "You are an expert financial analyst and economist. "
    "When given a financial or economic question, provide accurate, well-reasoned analysis. "
    "Show your calculations when relevant. Cite key financial concepts and metrics. "
    "Note: this is for educational purposes only — not financial advice."
)

# ~10% of samples should include the disclaimer in the response
# to prevent the model from dropping it — matches training distribution
FINANCE_DISCLAIMER = "\n\nNote: This analysis is for educational purposes only and does not constitute financial advice."


def format_finance_sample(sample: dict, idx: int) -> str:
    instruction = sample.get("instruction", "") or ""
    context     = sample.get("input", "") or ""
    response    = sample.get("output", "") or ""

    user_msg = instruction
    if context.strip():
        user_msg = f"{instruction}\n\nContext:\n{context}"

    # Add disclaimer to ~10% of samples
    if idx % 10 == 0:
        response = response + FINANCE_DISCLAIMER

    messages = [
        {"role": "system",    "content": FINANCE_SYSTEM},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": response},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


# Test formatter
formatted = format_finance_sample(raw[0], 0)
print(f"Example (first 600 chars):\n{formatted[:600]}")
print(f"\nToken count: {len(tokenizer.encode(formatted))}")

In [ ]:
split     = raw.train_test_split(test_size=TEST_SIZE, seed=42)
train_raw = split["train"]
test_raw  = split["test"]
print(f"Train: {len(train_raw)} | Test: {len(test_raw)}")

train_dataset = train_raw.map(
    lambda s, idx: {"text": format_finance_sample(s, idx)},
    with_indices=True,
    remove_columns=train_raw.column_names,
)
print(f"Formatted: {len(train_dataset)} training samples")

## 5. Load Model in 4-bit + Inject LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train

In [ ]:
import time
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=False,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    group_by_length=True,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
steps_per_epoch = len(train_dataset) // eff_batch
print(f"Training {len(train_dataset)} samples × {NUM_EPOCHS} epochs")
print(f"Effective batch: {eff_batch} | Steps/epoch: ~{steps_per_epoch}")

t0 = time.time()
trainer.train()
print(f"\nTraining time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
import os

adapter_path = f"{OUTPUT_DIR}/final-adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter saved: {adapter_path} ({total_size / 1e6:.1f} MB)")

## 7. Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

log_history  = trainer.state.log_history
train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e]

if train_losses:
    steps, losses = zip(*train_losses)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses, color='steelblue', linewidth=1.5, alpha=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('Training Loss')
    ax.set_title('Phase 4: Finance QLoRA — Training Loss')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=150)
    plt.show()
    print(f"Initial: {losses[0]:.4f} | Final: {losses[-1]:.4f} | Reduction: {(losses[0]-losses[-1])/losses[0]:.1%}")

## 8. Qualitative Inference Check

In [ ]:
model.eval()

def generate_finance(question: str, max_new_tokens: int = 350) -> str:
    messages = [
        {"role": "system", "content": FINANCE_SYSTEM},
        {"role": "user",   "content": question},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs    = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()


test_questions = [
    "Explain the relationship between bond prices and interest rates.",
    "A company has revenue of $10M, COGS of $6M, operating expenses of $2M, tax rate 25%. Calculate EBIT and net income.",
    "What is the difference between systematic and unsystematic risk?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate_finance(q)[:400]}")
    print()

## 9. LLM-as-Judge Evaluation (Groq)

Finance rubric: 5-point scale checking correct formula/concept, direction of effect, key caveats, appropriate risk flags.

In [ ]:
import re
try:
    from kaggle_secrets import UserSecretsClient
    groq_key = UserSecretsClient().get_secret("GROQ_API_KEY")
    print("GROQ_API_KEY loaded.")
except Exception:
    import os
    groq_key = os.environ.get("GROQ_API_KEY") or input("Enter GROQ_API_KEY: ").strip()

from groq import Groq
groq_client = Groq(api_key=groq_key)
JUDGE_MODEL = "llama-3.3-70b-versatile"

FINANCE_RUBRIC = (
    "Rating criteria for financial/economic responses:\n"
    "1 — Wrong formula, incorrect concept, or fundamentally misleading\n"
    "2 — Correct general direction but significant errors in calculation or reasoning\n"
    "3 — Correct answer but missing key caveats or incomplete calculation steps\n"
    "4 — Correct, shows calculation, mentions key assumptions; minor omissions\n"
    "5 — Accurate, complete calculation, correct interpretation, appropriate risk flags"
)

In [ ]:
from tqdm.auto import tqdm

eval_samples = test_raw.select(range(min(MAX_EVAL_SAMPLES, len(test_raw))))
scores, records = [], []

for sample in tqdm(eval_samples, desc="Finance LLM-judge"):
    instruction = sample.get("instruction", "") or ""
    context     = sample.get("input", "") or ""
    gt_response = sample.get("output", "") or ""

    user_prompt = f"{instruction}\n\nContext:\n{context}" if context.strip() else instruction
    response    = generate_finance(user_prompt)

    judge_prompt = (
        f"You are an expert evaluator. Rate the following AI response on a scale of 1-5.\n\n"
        f"Question: {instruction[:400]}\n\n"
        f"Reference answer: {gt_response[:400]}\n\n"
        f"AI response: {response[:400]}\n\n"
        f"{FINANCE_RUBRIC}\n\n"
        f"Respond with ONLY a single digit (1-5). No explanation."
    )

    try:
        completion = groq_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=5,
            temperature=0,
        )
        score_text = completion.choices[0].message.content or ""
        m = re.search(r'[1-5]', score_text.strip())
        score = int(m.group()) if m else 3
    except Exception as e:
        print(f"Judge error: {e}")
        score = 3

    scores.append(score)
    records.append({"instruction": instruction[:100], "score": score})

avg_score  = sum(scores) / len(scores) if scores else 0.0
score_dist = {str(i): scores.count(i) for i in range(1, 6)}

print(f"\n{'='*50}")
print(f"FINANCE LLM-JUDGE RESULTS")
print(f"Average score: {avg_score:.2f}/5.0")
print(f"Distribution:  {score_dist}")
print(f"Baseline:      {DOMAIN_BASELINE}")
print(f"Target:        {DOMAIN_TARGET}")
print(f"{'='*50}")

In [ ]:
import json

eval_summary = {
    "phase":        "phase4_finance",
    "model":        BASE_MODEL,
    "adapter":      adapter_path,
    "dataset":      DATASET_NAME,
    "num_samples":  MAX_SAMPLES,
    "lora_r":       LORA_R,
    "num_epochs":   NUM_EPOCHS,
    "avg_score":    round(avg_score, 3),
    "score_dist":   score_dist,
    "baseline":     DOMAIN_BASELINE,
    "target":       DOMAIN_TARGET,
}
with open(f"{OUTPUT_DIR}/eval_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)
print(json.dumps(eval_summary, indent=2))

## 10. Push Adapter to HuggingFace Hub

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN") or input("Enter HF_TOKEN: ").strip()

from huggingface_hub import login, HfApi
login(token=hf_token)

HF_USERNAME = "anksriv"
REPO_ID     = f"{HF_USERNAME}/mistral-7b-finance-qlora"

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True, private=False)

trainer.model.push_to_hub(REPO_ID, token=hf_token)
tokenizer.push_to_hub(REPO_ID, token=hf_token)
api.upload_file(
    path_or_fileobj=f"{OUTPUT_DIR}/eval_results.json",
    path_in_repo="eval_results.json",
    repo_id=REPO_ID,
    token=hf_token,
)

print(f"\nPushed: https://huggingface.co/{REPO_ID}")

## 11. Summary

| Item | Value |
|------|-------|
| Base model | `mistralai/Mistral-7B-Instruct-v0.2` |
| Dataset | `gbharti/finance-alpaca` (shuffled 10K) |
| LoRA rank | r=16 |
| Epochs | 3 |
| Max seq length | 512 |
| Eval metric | LLM-as-judge 1–5 (finance rubric) |
| Baseline | ~3.0/5.0 |
| Target | ≥3.5/5.0 |

**Dataset note**: `gbharti/finance-alpaca` contains date-sensitive data (historical prices, past rates).  
The system prompt instructs the model to flag such figures for verification — this is reinforced by including the disclaimer in 10% of training samples.

**Next step:** Run `notebooks/05_coding.ipynb`.